In [1]:
import xarray as xr
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import os, re
import pandas as pd
os.chdir(r'H:\CDMet\\')

In [53]:
CDMet_dir = r'H:\CDMet\Total_Precipitation'
nc_list = [f[:-3] for f in os.listdir(CDMet_dir) if f.endswith('.nc')]
csv_list = [f[:-4] for f in os.listdir('zonal_mean_df') if f.endswith('.csv')]
file_list =  [item for item in nc_list if item not in csv_list]

In [54]:
len(file_list)

21

In [58]:
index_df = pd.read_csv(('index_table.csv'))
index_df = index_df.groupby('abbre').apply(lambda x: x.sample(n=min(20, len(x)), random_state=1)).reset_index(drop=True)
variable = 'pre'

C:\Users\HP\AppData\Local\Temp\ipykernel_28380\1268642625.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  index_df = index_df.groupby('abbre').apply(lambda x: x.sample(n=min(20, len(x)), random_state=1)).reset_index(drop=True)


In [59]:
index_df

,Unnamed: 0,lon,lat,lon_index,lat_index,abbre
0,881889,85.395840,42.610222,609,408,dsk
1,862427,84.479173,42.985222,587,399,dsk
2,883998,83.270840,42.568555,558,409,dsk
3,862452,85.520840,42.985222,612,399,dsk
4,860279,84.979173,43.026888,599,398,dsk
...,...,...,...,...,...,...
135,907682,80.104173,42.110222,482,420,xhl
136,911996,79.854173,42.026888,476,422,xhl
137,924939,79.145840,41.776888,459,428,xhl
138,916282,78.437506,41.943555,442,424,xhl


In [60]:
for file_name in file_list:

    print(file_name)
    
    # index_df = pd.read_excel('lat_lon_index.xlsx',sheet_name=model_name)
    ds = xr.open_dataset(os.path.join(CDMet_dir, file_name+'.nc'))


    lon_indices = index_df['lon_index'].tolist()
    lat_indices = index_df['lat_index'].tolist()
    # Extract the 'pr' variable from the dataset using the indices
    # Assuming 'lon_index' and 'lat_index' are valid indices for your dataset
    pr_data = ds[variable].isel(lon=lon_indices, lat=lat_indices)
    pr_df = pr_data.to_dataframe().reset_index()
    pr_df['lat'] = pr_df['lat'].round(6)
    pr_df['lon'] = pr_df['lon'].round(6)
    index_df['lat'] = index_df['lat'].round(6)
    index_df['lon'] = index_df['lon'].round(6)
    pr_df_merged = pr_df.merge(index_df, on=['lat','lon'],how='left')
    pr_df_merged = pr_df_merged.dropna(how='any')
    pr_df_merged.drop(['lat_index','lon_index' ],axis=1,inplace=True)
    mean_pr_df = pr_df_merged.groupby(by=['time','abbre']).mean()
    # Pivot the DataFrame
    pr_df = mean_pr_df.reset_index().pivot(index='time', columns='abbre', values=variable)

    # Optional: Rename columns to make them more descriptive (if needed)
    pr_df.columns.name = None  # Remove the name of the columns index if undesired

    # Convert cftime.DatetimeNoLeap to pandas datetime
    pr_df.index = pr_df.index.map(lambda x: x.strftime('%Y-%m-%d') if hasattr(x, 'strftime') else x)
    pr_df.index = pd.to_datetime(pr_df.index)

    # Format time index as year-month
    pr_df.index = pr_df.index.to_period('D')
    # Display the resulting DataFrame
    pr_df.to_csv(os.path.join('zonal_mean_df', file_name+'.csv'))


CDMet_pre_2000
CDMet_pre_2001
CDMet_pre_2002
CDMet_pre_2003
CDMet_pre_2004
CDMet_pre_2005
CDMet_pre_2006
CDMet_pre_2007
CDMet_pre_2008
CDMet_pre_2009
CDMet_pre_2010
CDMet_pre_2011
CDMet_pre_2012
CDMet_pre_2013
CDMet_pre_2014
CDMet_pre_2015
CDMet_pre_2016
CDMet_pre_2017
CDMet_pre_2018
CDMet_pre_2019
CDMet_pre_2020


In [41]:
pr_df_merged

,time,lat,lon,meantmp,Unnamed: 0,abbre
0,1.0,43.318555,84.979173,258.30,845159.0,dsk
9,1.0,43.318555,84.979173,258.30,845159.0,dsk
94,1.0,43.110222,84.354173,259.44,855944.0,dsk
101,1.0,43.110222,84.354173,259.44,855944.0,dsk
188,1.0,43.068555,85.645840,259.39,858135.0,dsk
...,...,...,...,...,...,...
3165345,366.0,34.985222,79.895840,256.49,1277037.0,wlwt
3165408,366.0,34.901888,79.645840,256.16,1281351.0,wlwt
3165424,366.0,34.901888,79.645840,256.16,1281351.0,wlwt
3165439,366.0,34.901888,79.645840,256.16,1281351.0,wlwt


In [74]:
import pandas as pd
import glob

pre_file_pattern = 'zonal_mean_df\\CDMet_pre_*.csv'
tmp_file_pattern = 'zonal_mean_df\\CDMet_meantmp_*.csv'

pre_files = glob.glob(pre_file_pattern)
tmp_files = glob.glob(tmp_file_pattern)


pre_dataframes = [pd.read_csv(file) for file in pre_files]
pre_combined_df = pd.concat(pre_dataframes, ignore_index=True)
tmp_dataframes = [pd.read_csv(file) for file in tmp_files]
tmp_combined_df = pd.concat(tmp_dataframes, ignore_index=True)

pre_combined_df['time'] =pd.date_range(start='2000-01-01', end='2020-12-31', freq='D')
tmp_combined_df['time'] =pd.date_range(start='2000-01-01', end='2020-12-31', freq='D')

           time       dsk       hsg   kq  slglk  tgzlk  wlwt    xhl
0    2000-01-01  0.130556  0.096703  0.0  0.025    0.0   0.0  0.075
1    2000-01-02  1.340278  0.158242  0.0  0.000    0.0   0.0  0.000
2    2000-01-03  1.628056  0.323297  0.0  0.000    0.0   0.0  1.860
3    2000-01-04  0.000000  0.000000  0.0  0.000    0.0   0.0  0.000
4    2000-01-05  0.000000  0.000000  0.0  0.000    0.0   0.0  0.000
...         ...       ...       ...  ...    ...    ...   ...    ...
7666 2020-12-27  0.000000  0.000000  0.0  0.000    0.0   0.0  0.000
7667 2020-12-28  0.000000  0.000000  0.0  0.000    0.0   0.0  0.000
7668 2020-12-29  0.000000  0.000000  0.0  0.000    0.0   0.0  0.000
7669 2020-12-30  0.000000  0.000000  0.0  0.000    0.0   0.0  0.000
7670 2020-12-31  0.000000  0.000000  0.0  0.000    0.0   0.0  0.000

[7671 rows x 8 columns]
           time         dsk         hsg          kq      slglk       tgzlk  \
0    2000-01-01  260.550972  261.542747  265.566579  265.08875  263.246111   
1  

In [75]:
pre_combined_df.to_csv('CDMet_pr_combined_df.csv')
tmp_combined_df.to_csv('CDMet_tm_combined_df.csv')

# The follows is for get the index of each points in each basins and make a index table

In [11]:
ds

<xarray.Dataset> Size: 7GB
Dimensions:  (lon: 2160, lat: 1118, time: 366)
Coordinates:
  * lon      (lon) float64 17kB 60.02 60.06 60.1 60.15 ... 149.9 149.9 150.0
  * lat      (lat) float64 9kB 59.61 59.57 59.53 59.49 ... 13.15 13.11 13.07
  * time     (time) float32 1kB 1.0 2.0 3.0 4.0 5.0 ... 363.0 364.0 365.0 366.0
Data variables:
    meantmp  (time, lat, lon) float64 7GB ...

In [10]:
basin_gdf = gpd.read_file(r'F:\geodata\river_runoff_obs\Tarim.shp')

In [13]:
ds = xr.open_dataset(os.path.join(CDMet_dir, file_name+'.nc'))
lon_arr = ds.variables['lon'].values
lat_arr = ds.variables['lat'].values
lon_grid, lat_grid = np.meshgrid(lon_arr, lat_arr)

# Create points and track indices
points = []
lon_indices = []
lat_indices = []

for lat_idx, lat in enumerate(lat_arr):
    for lon_idx, lon in enumerate(lon_arr):
        points.append(Point(lon, lat))  # Create Point object
        lon_indices.append(lon_idx)    # Track longitude index
        lat_indices.append(lat_idx)    # Track latitude index

# Create GeoDataFrame
gdf = gpd.GeoDataFrame(geometry=points)
gdf['lon'] = gdf.geometry.x  # Extract longitude (X-coordinate)
gdf['lat'] = gdf.geometry.y  # Extract latitude (Y-coordinate)
gdf['lon_index'] = lon_indices  # Add longitude indices
gdf['lat_index'] = lat_indices  # Add latitude indices

# Check if CRS is set
if gdf.crs is None:
    # Set the CRS to WGS84 (latitude and longitude in degrees)
    gdf = gdf.set_crs("EPSG:4326")
gdf = gdf.to_crs(basin_gdf.crs)
# Keep only valid geometries
gdf = gdf[gdf.geometry.is_valid & ~gdf.geometry.is_empty]



In [16]:
joined_gdf = gpd.sjoin(gdf,basin_gdf,how='left')

In [18]:
station_gdf = joined_gdf.dropna(subset='abbre')

In [20]:
station_df = station_gdf[['lon','lat','lon_index','lat_index','abbre']]

In [23]:
station_df.to_csv('index_table.csv')

In [14]:
gdf.to_file(f'H:\CDMet\CDMet.shp', driver='ESRI Shapefile')
del gdf

<>:1: SyntaxWarning: invalid escape sequence '\C'
<>:1: SyntaxWarning: invalid escape sequence '\C'
C:\Users\HP\AppData\Local\Temp\ipykernel_28380\3088716857.py:1: SyntaxWarning: invalid escape sequence '\C'
  gdf.to_file(f'H:\CDMet\CDMet.shp', driver='ESRI Shapefile')
C:\Users\HP\AppData\Local\Temp\ipykernel_28380\3088716857.py:1: SyntaxWarning: invalid escape sequence '\C'
  gdf.to_file(f'H:\CDMet\CDMet.shp', driver='ESRI Shapefile')


KeyboardInterrupt: 